In [ ]:
!pip install docling

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 2.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.0/506.0 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 279.0/279.0 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.0/94.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.7/42.7 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 97.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 45.5 MB/s eta 0:00:00
   ━━━

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Example conversion from PDF to Markdown

In [ ]:
import logging
import time
from pathlib import Path
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive', force_remount=True)

from docling.backend.docling_parse_backend import DoclingParseDocumentBackend
from docling.datamodel.base_models import ConversionStatus, InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, AcceleratorOptions
from docling.document_converter import DocumentConverter, PdfFormatOption

# Set up logging
logging.basicConfig(level=logging.INFO)
_log = logging.getLogger(__name__)

def main():
    # --- PATH CONFIGURATION ---
    source_path = Path("/content/drive/MyDrive/NIST.AI.800-4.pdf")
    output_dir = Path("/content/drive/MyDrive/converted_markdown")

    image_output_dir = output_dir / "images"
    output_dir.mkdir(parents=True, exist_ok=True)
    image_output_dir.mkdir(parents=True, exist_ok=True)

    # 2. Configure Pipeline
    pipeline_options = PdfPipelineOptions()
    pipeline_options.do_ocr = True
    pipeline_options.do_table_structure = True
    pipeline_options.generate_page_images = True
    pipeline_options.images_scale = 2.0
    pipeline_options.accelerator_options = AcceleratorOptions(num_threads=2)

    doc_converter = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(
                pipeline_options=pipeline_options,
                backend=DoclingParseDocumentBackend
            )
        }
    )

    _log.info(f"--- Starting Conversion: {source_path.name} ---")
    start_time = time.time()

    try:
        conv_res = doc_converter.convert(source_path)

        if conv_res.status == ConversionStatus.SUCCESS:
            doc_filename = source_path.stem
            output_file = output_dir / f"{doc_filename}.md"
            md_output = conv_res.document.export_to_markdown()

            # 3. Process items and handle image references
            image_count = 0
            for element, _level in conv_res.document.iterate_items():
                if hasattr(element, "image"):
                    image_count += 1
                    # Define names even if extraction fails
                    image_id = getattr(element, "id_", image_count)
                    image_label = getattr(element, "label", "figure")
                    image_name = f"{image_label}_{image_id}.png"
                    image_file_path = image_output_dir / image_name

                    # --- THE FAIL-SAFE IMAGE REFERENCE ---
                    # We create this block regardless of whether the file save works
                    replacement_text = (
                        f"\n\n![{image_label}](images/{image_name})\n"
                        f"> **[TODO: MANUAL CAPTION FOR {image_name}]**\n"
                        f"> *Status:* {'[Image Extracted]' if element.image else '[EXTRACTION FAILED - CHECK PDF]'}\n"
                        f"> *Summary:* \n"
                        f"> *Key Keywords:* \n\n"
                    )

                    # Replace the placeholder in the MD text
                    default_placeholder = f"<!-- {image_label}_{image_id} -->"
                    if default_placeholder in md_output:
                        md_output = md_output.replace(default_placeholder, replacement_text)
                    else:
                        md_output = md_output.replace(f"[{image_label}]", replacement_text, 1)

                    # --- ATTEMPT PHYSICAL EXTRACTION ---
                    if element.image is not None:
                        try:
                            element.image.pil_image.save(image_file_path, format="PNG")
                        except Exception as img_err:
                            _log.warning(f"Could not save image file {image_name}: {img_err}")
                    else:
                        _log.warning(f"No image data found for {image_name}, reference kept in Markdown.")

            # Save the final Markdown file
            with output_file.open("w", encoding="utf-8") as fp:
                fp.write(md_output)

            _log.info(f"--- SUCCESS! ---")
            _log.info(f"Markdown: {output_file}")
            _log.info(f"Total images referenced: {image_count}")
        else:
            _log.error(f"Conversion failed with status: {conv_res.status}")

    except Exception as e:
        _log.error(f"--- SYSTEM ERROR: {e} ---")

    _log.info(f"Total time: {time.time() - start_time:.2f} seconds.")

if __name__ == "__main__":
    main()

Mounted at /content/drive


[INFO] 2026-05-13 13:48:20,243 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-05-13 13:48:20,244 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-05-13 13:48:20,259 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-05-13 13:48:20,261 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-05-13 13:48:20,429 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-05-13 13:48:20,431 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-05-13 13:48:20,432 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-05-13 13:48:20,434 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-05-13 13:48:20,508 [Ra

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

Clean data scripts